# RISE: software preflight

This notebook checks the Python environment without generating market paths. The frozen study settings are in [PROTOCOL.md](PROTOCOL.md). The completed pilot, ledgers, validation, and E1–E3 results are presented in [02_run_and_review.ipynb](02_run_and_review.ipynb).

## 1. Check which Python is running

Use the project environment when launching Jupyter. The next cell prints the interpreter and finds the project root.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "simulation":
    root = root.parent
assert (root / "simulation" / "check_environment.py").is_file(), "Open this notebook from the project folder."
print("Python:", sys.executable)
print("Project:", root)


## 2. Run the software checks

These checks compare Black–Scholes prices from separate implementations, invert a known price back to implied volatility, and examine a nearly constant-variance Heston diagnostic. Their inputs are software-test fixtures, not selected research parameters. A pass does not validate the later simulation and hedge-accounting engine.

In [ ]:
import subprocess
import json

check = subprocess.run(
    [sys.executable, str(root / "simulation" / "check_environment.py")],
    cwd=root, text=True, capture_output=True, timeout=90,
)
print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Software checks failed; inspect the report before continuing.")
report = json.loads((root / "outputs" / "setup" / "environment_check.json").read_text())


## 3. Inspect what passed

Each row is a software check. None is a comparison of the research hedge strategies.

In [ ]:
import pandas as pd

pd.DataFrame([{"check": name, "passed": passed} for name, passed in report["checks"].items()])


## 4. Review the completed pilot

The pilot configuration is fixed in [protocol.json](protocol.json). It distinguishes market-generating and pricing inputs and records the contract, time grid, sample size, seed, and accounting. Open [the review notebook](02_run_and_review.ipynb) to inspect the actual stock/variance path, holdings, and cash ledgers.

## 5. Continue to the completed study

[RESULTS_REPORT.md](RESULTS_REPORT.md) reports E1 (reference inputs), E2 (structural errors), and E3 (fees and rebalancing) across six contracts. It includes paired uncertainty, numerical checks, recoveries, and exclusions. The software checks above establish environment functionality; the separate study validations support the research calculations.